# Tools Basics

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangChain roadmap.

You will learn how to create tools with the `@tool` decorator and attach them to agents.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. The @tool Decorator

The simplest way to create a tool is with the `@tool` decorator.

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b

print("Name:", multiply.name)
print("Description:", multiply.description)
print("Schema:", multiply.args_schema.model_json_schema())

## 4. Type Hints Generate Schemas

LangChain reads your type hints to build the input schema the LLM sees.

In [ ]:
@tool
def search_products(query: str, max_results: int = 5) -> str:
    """Search for products matching the query."""
    return f"Found {max_results} products for '{query}'"

print("Schema:", search_products.args_schema.model_json_schema())

## 5. Docstrings as Descriptions

The docstring is the most important part — it tells the LLM when to use the tool.

In [ ]:
@tool
def get_stock_price(ticker: str) -> str:
    """Get the current stock price for a given ticker symbol.

    Use this when the user asks about stock prices, market values,
    or share prices for a specific company.
    """
    return f"${ticker}: $142.50"

print("Description:", get_stock_price.description)

## 6. Custom Tool Names

Override the default function name by passing a name to the decorator.

In [ ]:
@tool("stock_lookup")
def get_stock(ticker: str) -> str:
    """Look up the current price of a stock."""
    return f"${ticker}: $142.50"

print("Tool name:", get_stock.name)

## 7. Tool Return Types

Tools can return strings or dictionaries.

In [ ]:
@tool
def greet(name: str) -> str:
    """Greet a user by name."""
    return f"Hello, {name}!"

@tool
def get_user_info(user_id: str) -> dict:
    """Get information about a user."""
    return {
        "name": "Alice",
        "email": "alice@example.com",
        "plan": "premium",
    }

print("String return:", greet.invoke({"name": "Bob"}))
print("Dict return:", get_user_info.invoke({"user_id": "123"}))

## 8. Attaching Tools to an Agent

Pass tools to `create_react_agent` to give the agent access to them.

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

model = init_chat_model("gpt-4o-mini", model_provider="openai")

agent = create_react_agent(
    model=model,
    tools=[multiply, search_products, get_stock_price],
    prompt="You are a helpful assistant with access to several tools.",
)

result = agent.invoke({"messages": [HumanMessage(content="What is 15 times 23?")]})
print(result["messages"][-1].content)

## 9. Agent Picks the Right Tool

The agent selects the appropriate tool based on the user's question.

In [ ]:
result = agent.invoke({
    "messages": [HumanMessage(content="What's the stock price of AAPL?")]
})

for msg in result["messages"]:
    if msg.type == "tool":
        print(f"Tool used: {msg.name} -> {msg.content}")

print(f"\nFinal answer: {result['messages'][-1].content}")

## Summary

- The `@tool` decorator converts Python functions into LangChain tools
- Type hints generate input schemas automatically
- Docstrings become tool descriptions the LLM reads
- Tools can return strings or dicts
- Pass tools to `create_react_agent` for agent-powered workflows